In [1]:
import pandas as pd
import recordlinkage as rl
df_spotify = pd.read_csv("SpotifyCleaned.csv")
df_million_song = pd.read_csv("MillionSongCleaned.csv")

In [2]:
df_spotify.head()

,Unnamed: 0,artist,song,duration_ms,explicit,year,popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,genre
0,44,Missy Elliott,Hot Boyz,215466,True,1998,49,0.727,0.445,1,-11.241,1,0.2910,0.3390,0.000000,0.180,0.527,81.125,"hip hop, pop, R&B"
1,1,blink-182,All The Small Things,167066,False,1999,79,0.434,0.897,0,-4.918,1,0.0488,0.0103,0.000000,0.612,0.684,148.726,"rock, pop"
2,55,Mariah Carey,Thank God I Found You (feat. Joe & 98°),257360,False,1999,59,0.348,0.532,10,-5.882,1,0.0331,0.5920,0.000000,0.106,0.148,129.297,"pop, R&B"
3,2,Faith Hill,Breathe,250546,False,1999,66,0.529,0.496,7,-9.007,1,0.0290,0.1730,0.000000,0.251,0.278,136.859,"pop, country"
4,34,Britney Spears,Born to Make You Happy,243533,False,1999,58,0.633,0.922,11,-4.842,0,0.0454,0.1160,0.000465,0.071,0.686,84.110,pop


In [3]:
df_million_song.head()

,Unnamed: 0,SongNumber,SongID,AlbumID,AlbumName,ArtistID,ArtistName,Duration,KeySignature,KeySignatureConfidence,...,mbID,ArtistFamiliarity,Hotness,end_of_fade_in,key,keyConfidence,Loudness,mode,mode_confidence,start_of_fade_out
0,141,142,SOLOJYI12A8C13C49E,217883,Operaatio Jalokivimeri,AR22KLC1187B98E812,Kaija Koo,192.26077,11,0.436,...,446455f2-7d62-45ca-a7ef-53c1c676ac5c,0.500876,0.379228,2.067,11,0.436,-8.228,0,0.508,188.111
1,175,176,SOESZEO12AB017F908,347757,Pieni Maailma,ARLRHRM11E2835E415,Mandi,200.59383,7,0.762,...,5dc3323e-deec-4658-b87f-3488792df573,0.639569,0.295502,0.322,7,0.762,-6.572,1,0.957,194.815
2,9671,9672,SOFZXEV12A8C13A407,297039,Speed Ballads,AR0U44O1187B99007C,Republica,264.22812,5,0.572,...,78b8347b-a1e2-4e36-a917-92d632a62c18,0.614756,0.396490,0.136,5,0.572,-3.653,1,0.574,260.377
3,2898,2899,SOYXCOI12A8C1421AB,521002,Perfect World,AR3CLBX1187FB5A307,Richi M.,590.00118,1,0.160,...,c5a57ad4-1f29-43d4-9d7b-f7c2003d36ef,0.435448,0.291686,0.317,1,0.160,-8.729,1,0.229,585.886
4,2901,2902,SOVJSWO12A8C13426D,142324,Polkas de mi Tierra,ARSWXNX1187FB3D04E,Chango Spasiuk,36.20526,5,0.093,...,d30615cd-2e8f-4169-a879-fffa8b17f362,0.468251,0.410842,0.206,5,0.093,-19.650,1,0.343,36.205


In [5]:
df_spotify[["song", "artist"]]

,song,artist
0,Hot Boyz,Missy Elliott
1,All The Small Things,blink-182
2,Thank God I Found You (feat. Joe & 98°),Mariah Carey
3,Breathe,Faith Hill
4,Born to Make You Happy,Britney Spears
...,...,...
1800,Lips Of An Angel,Hinder
1801,Circles,Post Malone
1802,Options,NSG
1803,All The Things She Said,t.A.T.u.


In [6]:
df_million_song[["Title", "ArtistName"]]

,Title,ArtistName
0,Savu hälvenee,Kaija Koo
1,Pieni Maailma,Mandi
2,Pub Pusher,Republica
3,Wake Me Up,Richi M.
4,Autores Anonimos,Chango Spasiuk
...,...,...
2913,Dead Wrong,Cancer Bats
2914,Nothing To Lose,Boondox
2915,You And Your Heart,Jack Johnson
2916,The Vitalized Shell,Enthroned


In [7]:
# Make copies so we don't overwrite original
sp = df_spotify.copy()
ms = df_million_song.copy()

# Rename Million Song columns to match Spotify's
ms = ms.rename(columns={"ArtistName": "artist", "Title": "song"})

# Optional: strip whitespace & lowercase for cleaner matching
sp["artist_clean"] = sp["artist"].str.strip().str.lower()
sp["song_clean"]   = sp["song"].str.strip().str.lower()

ms["artist_clean"] = ms["artist"].str.strip().str.lower()
ms["song_clean"]   = ms["song"].str.strip().str.lower()

In [8]:
matches = sp.merge(
    ms,
    on=["artist_clean", "song_clean"],
    how="inner",
    suffixes=("_spotify", "_million")
)

In [9]:
result = matches[["artist_spotify", "song_spotify"]].drop_duplicates()
result

,artist_spotify,song_spotify
0,Britney Spears,Oops!...I Did It Again
1,Backstreet Boys,Shape of My Heart
2,Toni Braxton,He Wasn't Man Enough
3,Jennifer Lopez,Play
4,3 Doors Down,Here Without You
5,Maroon 5,This Love
6,Alicia Keys,Karma
7,Britney Spears,Everytime
8,Gwen Stefani,What You Waiting For?
9,Akon,Lonely


### Trying Again Using record Linkage

In [10]:
sp = df_spotify.copy()
ms = df_million_song.copy()

# Rename to consistent column names
sp = sp.rename(columns={"artist": "artist", "song": "song"})
ms = ms.rename(columns={"ArtistName": "artist", "Title": "song"})

# Clean text
for df in (sp, ms):
    df["artist_clean"] = df["artist"].str.lower().str.strip()
    df["song_clean"] = df["song"].str.lower().str.strip()

In [11]:
indexer = rl.Index()
indexer.block("artist_clean")   # Only compare rows with same artist spelling
candidate_links = indexer.index(sp, ms)

In [48]:
compare = rl.Compare()

compare.exact("artist_clean", "artist_clean", label="artist_exact")
compare.string("song_clean", "song_clean", method="jarowinkler", label="song_sim")

features = compare.compute(candidate_links, sp, ms)

In [69]:
fuzzy_matches = features[
    (features["song_sim"] > 0.86)
]

In [70]:
pairs = fuzzy_matches.index.tolist()

match_rows = []
for (i_sp, i_ms) in pairs:
    match_rows.append({
        "artist": sp.loc[i_sp, "artist"],
        "song": sp.loc[i_sp, "song"]
    })

result_df = pd.DataFrame(match_rows).drop_duplicates()
result_df

,artist,song
0,Britney Spears,Oops!...I Did It Again
1,Backstreet Boys,Shape of My Heart
2,Toni Braxton,He Wasn't Man Enough
3,Jennifer Lopez,Play
4,3 Doors Down,Here Without You
5,Maroon 5,This Love
6,Alicia Keys,Karma
7,Britney Spears,Everytime
8,Black Eyed Peas,Let's Get It Started - Spike Mix
9,Gwen Stefani,What You Waiting For?


In [71]:
pairs = fuzzy_matches.index.tolist()

rows = []
for (i_sp, i_ms) in pairs:
    rows.append({
        "spotify_artist": sp.loc[i_sp, "artist"],
        "spotify_song": sp.loc[i_sp, "song"],
        "million_artist": ms.loc[i_ms, "artist"],
        "million_song": ms.loc[i_ms, "song"],
        "similarity": fuzzy_matches.loc[(i_sp, i_ms), "song_sim"]
    })

matches_df = pd.DataFrame(rows)

In [72]:
matches_df

,spotify_artist,spotify_song,million_artist,million_song,similarity
0,Britney Spears,Oops!...I Did It Again,Britney Spears,Oops!...I Did It Again,1.000000
1,Backstreet Boys,Shape of My Heart,Backstreet Boys,Shape Of My Heart,1.000000
2,Toni Braxton,He Wasn't Man Enough,Toni Braxton,He Wasn't Man Enough,1.000000
3,Jennifer Lopez,Play,Jennifer Lopez,Play,1.000000
4,3 Doors Down,Here Without You,3 Doors Down,Here Without You,1.000000
5,Maroon 5,This Love,Maroon 5,This Love,1.000000
6,Alicia Keys,Karma,Alicia Keys,Karma,1.000000
7,Britney Spears,Everytime,Britney Spears,Everytime,1.000000
8,Black Eyed Peas,Let's Get It Started - Spike Mix,Black Eyed Peas,Let's Get It Started,0.925000
9,Gwen Stefani,What You Waiting For?,Gwen Stefani,What You Waiting For?,1.000000
